# Phase Voice: Embed Qwen2.5-Omni into Flux-Apex

**Transform FLUX into a self-contained multimodal AI system**

This notebook:
1. Downloads Flux-Apex-V1.flx from HuggingFace
2. Downloads and quantizes Qwen2.5-Omni-7B (text + audio + vision)
3. Embeds the voice module directly into .flx
4. **Removes** the byte decoder (not legacy — completely strips it)
5. Updates to version 5.0-voice-embedded
6. Saves back to Flux-Apex-V1.flx

**Why this matters:**
- No external LLM downloads at runtime
- Full any-to-any multimodal (text ↔ audio ↔ vision)
- Single self-contained .flx file
- True "voice" for FLUX architecture

**Expected sizes:**
- Current Flux-Apex-V1.flx: ~5.79 GB
- Decoder removed: -124 MB
- Qwen-Omni (4-bit) added: +2.8 GB
- **Final size: ~8.5 GB**

---

## Cell 1: Clone/Pull FLUX Repository

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')

# For local development
if not Path('/kaggle').exists() and not Path('/content').exists():
    ROOT = Path.cwd()
    if (ROOT / 'flux_utils.py').exists():
        print(f'  Using local directory: {ROOT}')
    else:
        ROOT = Path('/Users/admin/Desktop/flux')

if ROOT.exists() and (ROOT / '.git').exists():
    print('  Pulling latest...')
    os.chdir(ROOT)
    !git pull --ff-only 2>/dev/null || echo '  (pull skipped)'
elif not ROOT.exists():
    print('  Cloning repo...')
    !git clone {REPO_URL} {ROOT}
    os.chdir(ROOT)
else:
    os.chdir(ROOT)

print(f'  Working dir: {os.getcwd()}')

# Install dependencies
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt 2>/dev/null || echo '  (deps already installed)'
!pip install -q huggingface_hub transformers bitsandbytes accelerate 2>/dev/null

print('  ✓ Dependencies ready')

## Cell 2: Setup Paths & Logger

In [ ]:
import sys
from pathlib import Path

# Determine ROOT
if Path('/kaggle/working/FLUX').exists():
    ROOT = Path('/kaggle/working/FLUX')
elif Path('/content/FLUX').exists():
    ROOT = Path('/content/FLUX')
else:
    ROOT = Path.cwd()
    if not (ROOT / 'flux_utils.py').exists():
        ROOT = Path('/Users/admin/Desktop/flux')

ROOT_DIR = ROOT
PHASES_DIR = ROOT / 'phases'
CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Add paths
for p in [ROOT, PHASES_DIR / 'phase2', PHASES_DIR / 'phase_voice']:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from flux_utils import PhaseLogger, get_device, upload_flx_to_hf

log = PhaseLogger(phase=99)  # Phase Voice
log.separator("Phase Voice: Embed Qwen2.5-Omni into Flux-Apex")

print(f'  ROOT: {ROOT}')
print(f'  Checkpoints: {CHECKPOINT_DIR}')
print('  ✓ Paths configured')

## Cell 3: Hardware & Secrets

In [ ]:
import torch
import os

log.cell_start("Cell 3 — Hardware & Secrets")

device = get_device()
print(f'  Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {gpu_name} ({vram:.1f} GB)')
    
    # Check if enough VRAM for Qwen-Omni
    if vram < 15:
        print(f'  ⚠ Low VRAM ({vram:.1f} GB) — Qwen-Omni requires ~16GB for loading')
        print(f'    Will use aggressive quantization or CPU offload')
elif device == 'mps':
    print(f'  Using Apple Silicon MPS')
else:
    print(f'  ⚠ CPU only — this will be SLOW')

# Load HF token
hf_token = None

# Try Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    print('  ✓ Kaggle secrets loaded')
except:
    pass

# Try Colab secrets
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        print('  ✓ Colab secrets loaded')
    except:
        pass

# Environment variable
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('  ✓ Environment HF_TOKEN loaded')

if hf_token:
    hf_token = hf_token.strip()
    os.environ['HF_TOKEN'] = hf_token
else:
    print('  ⚠ No HF_TOKEN found — some downloads may fail')

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

## Cell 4: Download Flux-Apex-V1.flx

In [ ]:
from huggingface_hub import hf_hub_download

log.cell_start("Cell 4 — Download Flux-Apex")

APEX_PATH = CHECKPOINT_DIR / 'Flux-Apex-V1.flx'

# Check local first
if APEX_PATH.exists():
    size_gb = APEX_PATH.stat().st_size / 1e9
    print(f'  ✓ Found local: {APEX_PATH.name} ({size_gb:.2f} GB)')
else:
    print(f'  Downloading Flux-Apex-V1.flx from HuggingFace...')
    try:
        downloaded = hf_hub_download(
            repo_id='UnseenGAP/FLUX',
            filename='checkpoints/Flux-Apex-V1.flx',
            local_dir=str(ROOT_DIR),
            token=hf_token,
        )
        size_gb = Path(downloaded).stat().st_size / 1e9
        print(f'  ✓ Downloaded: {size_gb:.2f} GB')
    except Exception as e:
        print(f'  ✗ Download failed: {e}')
        raise

# Verify
assert APEX_PATH.exists(), f"Flux-Apex not found at {APEX_PATH}"

log.metric("Flux-Apex size", f"{APEX_PATH.stat().st_size / 1e9:.2f} GB")
log.cell_end("Cell 4 — Download Flux-Apex", "PASS")

## Cell 5: Analyze Current Flux-Apex Structure

In [ ]:
import torch
from datetime import datetime

log.cell_start("Cell 5 — Analyze Flux-Apex")

print(f'\n  Loading {APEX_PATH.name} for analysis...')
apex = torch.load(str(APEX_PATH), map_location='cpu', weights_only=False)

print(f'\n  ═══ CURRENT FLUX-APEX STRUCTURE ═══')
print(f'  Format: {apex.get("format", "unknown")}')
print(f'  Version: {apex.get("version", "unknown")}')
print(f'  Phase: {apex.get("phase", "unknown")}')
print(f'  Top-level keys: {len(apex)}')

# Analyze components to remove
print(f'\n  Components to REMOVE:')

# Decoder
if 'decoder' in apex:
    decoder_size = 0
    if isinstance(apex['decoder'], dict):
        for k, v in apex['decoder'].items():
            if isinstance(v, dict):
                for kk, vv in v.items():
                    if isinstance(vv, torch.Tensor):
                        decoder_size += vv.numel() * vv.element_size()
            elif isinstance(v, torch.Tensor):
                decoder_size += v.numel() * v.element_size()
    print(f'    ✓ decoder: {decoder_size / 1e6:.1f} MB')
else:
    print(f'    ✗ decoder: not found')

# LLM reference
if 'llm' in apex:
    print(f'    ✓ llm: external reference to be removed')
if 'llm_reference' in apex:
    print(f'    ✓ llm_reference: external reference to be removed')

# Check for WaveToText (failed experiment)
wave_to_text_found = False
if 'adapters' in apex:
    adapters = apex['adapters']
    if isinstance(adapters, dict):
        for key in adapters.keys():
            if 'text' in key.lower() and 'wave' in key.lower():
                wave_to_text_found = True
                print(f'    ✓ adapters.{key}: failed experiment to remove')

if not wave_to_text_found:
    print(f'    ✓ WaveToText: not present (good - was failed experiment)')

# Components to keep
print(f'\n  Components to KEEP:')
keep_components = ['cse', 'field', 'memory', 'bridges', 'causal', 'adapters', 
                   'spatial_memory', 'causal_tracker', 'learned_rules', 'grid_to_wave']
for comp in keep_components:
    if comp in apex:
        print(f'    ✓ {comp}')
    else:
        print(f'    ✗ {comp}: not found')

log.cell_end("Cell 5 — Analyze Flux-Apex", "PASS")

## Cell 6: Download and Quantize Qwen2.5-Omni-7B

This cell downloads the full Qwen-Omni model (~22GB) and quantizes it to 4-bit (~2.8GB).

**Requirements:**
- ~30GB disk space temporarily
- ~16GB VRAM (or CPU with patience)
- ~20-30 minutes on first run

In [ ]:
%%time
log.cell_start("Cell 6 — Quantize Qwen-Omni")

VOICE_PATH = CHECKPOINT_DIR / 'qwen_omni_4bit.pt'
MODEL_ID = 'Qwen/Qwen2.5-Omni-7B'

# Check if already quantized
if VOICE_PATH.exists():
    size_gb = VOICE_PATH.stat().st_size / 1e9
    print(f'  ✓ Found existing: {VOICE_PATH.name} ({size_gb:.2f} GB)')
    print(f'    Skipping quantization...')
    
    # Quick verify
    voice_state = torch.load(str(VOICE_PATH), map_location='cpu', weights_only=False)
    print(f'    Format: {voice_state.get("format", "unknown")}')
    print(f'    Thinker weights: {len(voice_state.get("thinker", {}))}')
    print(f'    Talker weights: {len(voice_state.get("talker", {}))}')

else:
    print(f'  Downloading and quantizing {MODEL_ID}...')
    print(f'  This will take 20-30 minutes and require ~30GB disk space.')
    print(f'  Coffee time! ☕\n')
    
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from transformers import BitsAndBytesConfig
        
        # Configure 4-bit quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        
        print(f'  [1/4] Loading model with 4-bit quantization...')
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            token=hf_token,
        )
        print(f'  ✓ Model loaded')
        
        print(f'  [2/4] Loading tokenizer...')
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_ID,
            trust_remote_code=True,
            token=hf_token,
        )
        print(f'  ✓ Tokenizer loaded')
        
        print(f'  [3/4] Extracting state dict...')
        state_dict = model.state_dict()
        total_params = sum(p.numel() for p in model.parameters())
        print(f'    Total parameters: {total_params:,}')
        print(f'    State dict keys: {len(state_dict)}')
        
        # Organize by module
        voice_state = {
            'format': 'FLUX_VOICE',
            'version': '1.0',
            'base_model': MODEL_ID,
            'quantization': '4bit',
            'timestamp': datetime.now().isoformat(),
            'total_params': total_params,
            
            'config': {
                'model_config': model.config.to_dict() if hasattr(model.config, 'to_dict') else {},
            },
            
            'thinker': {},
            'talker': {},
            'token2wav': {},
            
            'tokenizer': {
                'vocab_size': tokenizer.vocab_size if hasattr(tokenizer, 'vocab_size') else len(tokenizer),
                'special_tokens': dict(tokenizer.special_tokens_map) if hasattr(tokenizer, 'special_tokens_map') else {},
            },
        }
        
        # Sort weights into modules
        for key, tensor in state_dict.items():
            cpu_tensor = tensor.cpu()
            if key.startswith('thinker.'):
                voice_state['thinker'][key] = cpu_tensor
            elif key.startswith('talker.'):
                voice_state['talker'][key] = cpu_tensor
            elif key.startswith('token2wav.'):
                voice_state['token2wav'][key] = cpu_tensor
            else:
                # Default to thinker
                voice_state['thinker'][key] = cpu_tensor
        
        print(f'    Thinker weights: {len(voice_state["thinker"])}')
        print(f'    Talker weights: {len(voice_state["talker"])}')
        print(f'    Token2Wav weights: {len(voice_state["token2wav"])}')
        
        # Try to save tokenizer vocab
        try:
            voice_state['tokenizer']['vocab'] = tokenizer.get_vocab()
            print(f'    Vocab size: {len(voice_state["tokenizer"]["vocab"])}')
        except:
            print(f'    ⚠ Could not extract vocab')
        
        print(f'  [4/4] Saving to {VOICE_PATH}...')
        torch.save(voice_state, str(VOICE_PATH))
        
        size_gb = VOICE_PATH.stat().st_size / 1e9
        print(f'  ✓ Saved: {VOICE_PATH.name} ({size_gb:.2f} GB)')
        
        # Clean up GPU memory
        del model, state_dict
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    except ImportError as e:
        print(f'  ✗ Missing dependency: {e}')
        print(f'    Run: pip install transformers bitsandbytes accelerate')
        raise
    except Exception as e:
        print(f'  ✗ Quantization failed: {e}')
        raise

log.metric("Voice module size", f"{VOICE_PATH.stat().st_size / 1e9:.2f} GB")
log.cell_end("Cell 6 — Quantize Qwen-Omni", "PASS")

## Cell 7: Build New Flux-Apex with Embedded Voice

This is the core transformation:
1. Copy all FLUX core components (cse, field, memory, bridges, etc.)
2. **Remove** decoder completely (not just legacy flag — DELETE it)
3. **Remove** llm and llm_reference
4. **Add** voice module (Qwen-Omni weights)
5. Update version to 5.0-voice-embedded

In [ ]:
log.cell_start("Cell 7 — Build New Flux-Apex")

from datetime import datetime

# Load both models
print(f'\n  Loading source models...')
apex = torch.load(str(APEX_PATH), map_location='cpu', weights_only=False)
voice = torch.load(str(VOICE_PATH), map_location='cpu', weights_only=False)

original_size = APEX_PATH.stat().st_size
voice_size = VOICE_PATH.stat().st_size

print(f'  ✓ Flux-Apex loaded: {original_size / 1e9:.2f} GB')
print(f'  ✓ Voice module loaded: {voice_size / 1e9:.2f} GB')

# ─────────────────────────────────────────────
# STEP 1: Build new model structure
# ─────────────────────────────────────────────

print(f'\n  Building new Flux-Apex structure...')

new_apex = {
    # Header
    'format': 'FLUX',
    'version': '5.0-voice-embedded',
    'phase': 'phase_voice',
    'timestamp': datetime.now().isoformat(),
    'can_continue_learning': True,
}

# ─────────────────────────────────────────────
# STEP 2: Copy core FLUX components
# ─────────────────────────────────────────────

core_components = [
    'cse',           # Continuous Semantic Encoder
    'field',         # Resonance Field
    'memory',        # Three-tier memory
    'bridges',       # Wave ↔ Field projections
    'causal',        # CGN + causal graph
    'causal_tracker',
    'learned_rules',
    'spatial_memory',
    'grid_to_wave',
    'adapters',      # Grid, image, audio adapters
]

print(f'\n  Copying core components:')
for comp in core_components:
    if comp in apex:
        new_apex[comp] = apex[comp]
        print(f'    ✓ {comp}')
    else:
        print(f'    ✗ {comp}: not found in source')

# ─────────────────────────────────────────────
# STEP 3: REMOVE decoder completely
# ─────────────────────────────────────────────

print(f'\n  Removing old generation components:')

removed_components = []

# Decoder - REMOVE completely
if 'decoder' in apex:
    decoder_params = 0
    if isinstance(apex['decoder'], dict) and 'state_dict' in apex['decoder']:
        for v in apex['decoder']['state_dict'].values():
            if isinstance(v, torch.Tensor):
                decoder_params += v.numel()
    print(f'    ✓ decoder: REMOVED ({decoder_params:,} params, ~{decoder_params * 4 / 1e6:.1f} MB)')
    removed_components.append('decoder')
    # NOT copying to new_apex

# LLM - REMOVE completely
if 'llm' in apex:
    print(f'    ✓ llm: REMOVED (was external reference)')
    removed_components.append('llm')

# LLM reference - REMOVE completely  
if 'llm_reference' in apex:
    print(f'    ✓ llm_reference: REMOVED (was external reference)')
    removed_components.append('llm_reference')

# grid_adapters (legacy duplicate)
if 'grid_adapters' in apex:
    print(f'    ✓ grid_adapters: REMOVED (legacy duplicate of grid_to_wave)')
    removed_components.append('grid_adapters')

# ─────────────────────────────────────────────
# STEP 4: ADD voice module
# ─────────────────────────────────────────────

print(f'\n  Adding voice module (Qwen2.5-Omni-7B, 4-bit):')

new_apex['voice'] = {
    'format_version': '1.0',
    'base_model': voice.get('base_model', 'Qwen/Qwen2.5-Omni-7B'),
    'quantization': voice.get('quantization', '4bit'),
    'timestamp': datetime.now().isoformat(),
    'total_params': voice.get('total_params', 0),
    
    'config': voice.get('config', {}),
    'tokenizer': voice.get('tokenizer', {}),
    
    # Weights
    'thinker': voice.get('thinker', {}),
    'talker': voice.get('talker', {}),
    'token2wav': voice.get('token2wav', {}),
    
    # Bridge configs (initialized at runtime)
    'bridges': {
        'wave_to_voice': {'wave_dim': 432, 'voice_dim': 3584},
        'voice_to_wave': {'voice_dim': 3584, 'wave_dim': 432},
    },
}

print(f'    ✓ voice.thinker: {len(new_apex["voice"]["thinker"])} weights')
print(f'    ✓ voice.talker: {len(new_apex["voice"]["talker"])} weights')
print(f'    ✓ voice.token2wav: {len(new_apex["voice"]["token2wav"])} weights')
print(f'    ✓ voice.tokenizer: vocab_size={new_apex["voice"]["tokenizer"].get("vocab_size", "unknown")}')

# ─────────────────────────────────────────────
# STEP 5: Update components flags
# ─────────────────────────────────────────────

print(f'\n  Updating component flags:')

new_apex['components'] = {
    # Perception
    'cse': True,
    'grid_to_wave': True,
    'spatial_memory': True,
    'perception_field': False,
    
    # Knowledge
    'field': True,
    'working_memory': True,
    'episodic_memory': True,
    'semantic_memory': True,
    
    # Generation — NEW: Voice replaces decoder/llm
    'voice': True,
    'voice_thinker': True,
    'voice_talker': True,
    'voice_token2wav': True,
    
    # Generation — REMOVED
    'decoder': False,   # REMOVED
    'llm': False,       # REMOVED
    
    # Reasoning
    'causal_tracker': True,
    'rule_inducer': False,
    'goal_planner': False,
    'causal_graph': True,
    
    # Bridges & Adapters
    'bridges': True,
    'adapters': True,
    'learned_rules': True,
}

active = sum(1 for v in new_apex['components'].values() if v)
print(f'    Active components: {active}')

# ─────────────────────────────────────────────
# STEP 6: Update runtime_config
# ─────────────────────────────────────────────

print(f'\n  Updating runtime config:')

# Start fresh or copy existing
if 'runtime_config' in apex:
    new_apex['runtime_config'] = apex['runtime_config'].copy()
else:
    new_apex['runtime_config'] = {}

# Update generation config for voice
new_apex['runtime_config']['generation'] = {
    'voice_primary': True,             # NEW: Use embedded voice
    'llm_primary': False,              # DISABLED
    'byte_decoder_enabled': False,     # REMOVED
    'byte_decoder_learns_from_llm': False,  # N/A
    'generation_mode': 'voice',        # 'voice' | 'hybrid'
}

# Add voice config
new_apex['runtime_config']['voice'] = {
    'enabled': True,
    'model_type': 'qwen_omni',
    'quantization': '4bit',
    'max_tokens': 512,
    'temperature': 0.7,
    'top_p': 0.9,
    'text_enabled': True,
    'audio_input_enabled': True,
    'audio_output_enabled': True,
    'vision_enabled': True,
    'use_flux_context': True,
    'flux_context_limit': 10,
    'store_to_field': True,
}

print(f'    ✓ generation.voice_primary = True')
print(f'    ✓ voice config added')

# ─────────────────────────────────────────────
# STEP 7: Update metadata
# ─────────────────────────────────────────────

print(f'\n  Updating metadata:')

old_metadata = apex.get('metadata', {})

new_apex['metadata'] = {
    # Preserve lineage
    'base': old_metadata.get('base', 'Flux-Apex-V1.flx'),
    'created': old_metadata.get('created', datetime.now().isoformat()),
    
    # Update for this version
    'last_modified': datetime.now().isoformat(),
    'modified_components': ['voice', 'decoder', 'llm', 'llm_reference'],
    'removed_components': removed_components,
    
    # New capabilities
    'voice_embedded': True,
    'voice_base_model': 'Qwen/Qwen2.5-Omni-7B',
    'voice_quantization': '4bit',
    'no_external_downloads': True,
    
    # Capabilities list
    'capabilities': [
        'text',
        'grid',
        'image',
        'audio',
        'vision',
        'voice',             # NEW
        'speech_synthesis',  # NEW
        'audio_understanding',  # NEW
    ],
    
    # Phase info
    'phase': 'voice',
    'description': 'FLUX with embedded Qwen2.5-Omni voice module — self-contained multimodal AI',
}

# Copy preserved metadata
for key in ['cse_test', 'gtw_test', 'field_test', 'memory_test']:
    if key in old_metadata:
        new_apex['metadata'][key] = old_metadata[key]

print(f'    ✓ Metadata updated')
print(f'    ✓ Capabilities: {new_apex["metadata"]["capabilities"]}')

# Mark as modified
new_apex['modified'] = True
new_apex['modified_components'] = ['voice']

log.cell_end("Cell 7 — Build New Flux-Apex", "PASS")

## Cell 8: Verify New Structure

In [ ]:
log.cell_start("Cell 8 — Verify Structure")

print(f'\n  ═══ NEW FLUX-APEX STRUCTURE ═══')
print(f'  Version: {new_apex["version"]}')
print(f'  Phase: {new_apex["phase"]}')

print(f'\n  Top-level keys ({len(new_apex)}):')
for key in sorted(new_apex.keys()):
    if key == 'voice':
        print(f'    ✓ {key} (NEW - embedded Qwen-Omni)')
    elif key in ['decoder', 'llm', 'llm_reference', 'grid_adapters']:
        print(f'    ✗ {key} (REMOVED)')
    else:
        print(f'    • {key}')

# Verify critical components
print(f'\n  Critical components check:')
critical = ['cse', 'field', 'memory', 'bridges', 'voice']
all_present = True
for comp in critical:
    if comp in new_apex:
        print(f'    ✓ {comp}: present')
    else:
        print(f'    ✗ {comp}: MISSING!')
        all_present = False

# Verify removed components
print(f'\n  Removed components check:')
removed = ['decoder', 'llm', 'llm_reference', 'grid_adapters']
for comp in removed:
    if comp in new_apex:
        print(f'    ✗ {comp}: should be removed!')
        all_present = False
    else:
        print(f'    ✓ {comp}: removed')

if all_present:
    print(f'\n  ✓ Structure verification PASSED')
else:
    print(f'\n  ✗ Structure verification FAILED')

log.cell_end("Cell 8 — Verify Structure", "PASS" if all_present else "FAIL")

## Cell 9: Save New Flux-Apex-V1.flx

In [ ]:
log.cell_start("Cell 9 — Save Flux-Apex")

# Backup original (optional)
BACKUP_PATH = CHECKPOINT_DIR / 'Flux-Apex-V1-backup-v4.flx'
if not BACKUP_PATH.exists():
    import shutil
    print(f'  Creating backup: {BACKUP_PATH.name}')
    shutil.copy(APEX_PATH, BACKUP_PATH)
    print(f'  ✓ Backup created')
else:
    print(f'  Backup already exists: {BACKUP_PATH.name}')

# Save new model
print(f'\n  Saving new Flux-Apex-V1.flx...')
print(f'  (This will take a minute — lots of data to write)\n')

torch.save(new_apex, str(APEX_PATH))

new_size = APEX_PATH.stat().st_size
size_change = new_size - original_size

print(f'  ✓ Saved: {APEX_PATH.name}')
print(f'\n  Size comparison:')
print(f'    Original: {original_size / 1e9:.2f} GB')
print(f'    New:      {new_size / 1e9:.2f} GB')
print(f'    Change:   {size_change / 1e9:+.2f} GB ({size_change / original_size * 100:+.1f}%)')

log.metric("New file size", f"{new_size / 1e9:.2f} GB")
log.metric("Size change", f"{size_change / 1e9:+.2f} GB")

log.cell_end("Cell 9 — Save Flux-Apex", "PASS")

## Cell 10: Verify Saved Model

In [ ]:
log.cell_start("Cell 10 — Verify Saved Model")

print(f'\n  Reloading saved model for verification...')

# Clear memory
del new_apex
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Reload
verified = torch.load(str(APEX_PATH), map_location='cpu', weights_only=False)

print(f'\n  ═══ VERIFIED FLUX-APEX ═══')
print(f'  Format: {verified.get("format")}')
print(f'  Version: {verified.get("version")}')
print(f'  Phase: {verified.get("phase")}')

# Check voice module
print(f'\n  Voice module:')
if 'voice' in verified:
    voice = verified['voice']
    print(f'    ✓ Base model: {voice.get("base_model")}')
    print(f'    ✓ Quantization: {voice.get("quantization")}')
    print(f'    ✓ Thinker weights: {len(voice.get("thinker", {}))}')
    print(f'    ✓ Talker weights: {len(voice.get("talker", {}))}')
    print(f'    ✓ Token2Wav weights: {len(voice.get("token2wav", {}))}')
else:
    print(f'    ✗ Voice module not found!')

# Check removed components
print(f'\n  Removed components:')
for comp in ['decoder', 'llm', 'llm_reference', 'grid_adapters']:
    if comp in verified:
        print(f'    ✗ {comp}: still present (ERROR)')
    else:
        print(f'    ✓ {comp}: removed')

# Check components flags
print(f'\n  Component flags:')
comps = verified.get('components', {})
print(f'    voice: {comps.get("voice", False)}')
print(f'    decoder: {comps.get("decoder", False)}')
print(f'    llm: {comps.get("llm", False)}')

# Final verdict
verification_passed = (
    verified.get('format') == 'FLUX' and
    verified.get('version') == '5.0-voice-embedded' and
    'voice' in verified and
    'decoder' not in verified and
    'llm' not in verified
)

if verification_passed:
    print(f'\n  ═══ VERIFICATION PASSED ═══')
    log.success("Model verification passed!")
else:
    print(f'\n  ═══ VERIFICATION FAILED ═══')
    log.error("Model verification failed!")

log.cell_end("Cell 10 — Verify Saved Model", "PASS" if verification_passed else "FAIL")

## Cell 11: Upload to HuggingFace Hub

In [ ]:
log.cell_start("Cell 11 — Upload to HuggingFace")

if hf_token:
    print(f'\n  Uploading to HuggingFace Hub...')
    print(f'  File: {APEX_PATH.name} ({APEX_PATH.stat().st_size / 1e9:.2f} GB)')
    print(f'  This may take several minutes...\n')
    
    try:
        success = upload_flx_to_hf(
            str(APEX_PATH),
            hf_token=hf_token,
        )
        
        if success:
            log.success("Uploaded to HuggingFace Hub!")
            print(f'\n  ✓ View at: https://huggingface.co/UnseenGAP/FLUX')
        else:
            log.warning("Upload failed")
    except Exception as e:
        print(f'  ✗ Upload error: {e}')
        log.warning(f"Upload error: {str(e)[:50]}")
else:
    print(f'  ⚠ No HF_TOKEN — skipping upload')
    print(f'  Model saved locally at: {APEX_PATH}')

log.cell_end("Cell 11 — Upload to HuggingFace", "PASS")

## Cell 12: Update Documentation

In [ ]:
log.cell_start("Cell 12 — Update Documentation")

# Update FLUX_APEX_V1.md
docs_path = ROOT / 'DOCS' / 'FLUX_APEX_V1.md'

if docs_path.exists():
    print(f'  Updating {docs_path.name}...')
    
    # Read current
    content = docs_path.read_text()
    
    # Check if already updated for v5.0
    if '5.0-voice-embedded' in content:
        print(f'  ✓ Already updated for v5.0')
    else:
        print(f'  ⚠ Documentation needs manual update to v5.0')
        print(f'    Key changes:')
        print(f'    - Version: 4.0 → 5.0-voice-embedded')
        print(f'    - Added: voice component (Qwen2.5-Omni-7B)')
        print(f'    - Removed: decoder, llm, llm_reference')
        print(f'    - No external downloads required')
else:
    print(f'  ⚠ Documentation file not found: {docs_path}')

log.cell_end("Cell 12 — Update Documentation", "PASS")

## Cell 13: Summary

In [ ]:
log.separator("Phase Voice Complete")

original_gb = original_size / 1e9
new_gb = APEX_PATH.stat().st_size / 1e9
change_gb = new_gb - original_gb

print(f"""
╔════════════════════════════════════════════════════════════════════╗
║                    PHASE VOICE COMPLETE                            ║
║              FLUX IS NOW SELF-CONTAINED                            ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  WHAT CHANGED:                                                     ║
║  ├── Version: 4.0 → 5.0-voice-embedded                            ║
║  ├── ADDED: voice module (Qwen2.5-Omni-7B, 4-bit quantized)       ║
║  │   ├── Thinker: Text + Audio + Vision reasoning                 ║
║  │   ├── Talker: Speech synthesis                                 ║
║  │   └── Token2Wav: Vocoder                                       ║
║  └── REMOVED completely:                                          ║
║      ├── decoder (byte-level, replaced by voice)                  ║
║      ├── llm (external reference)                                 ║
║      ├── llm_reference (external reference)                       ║
║      └── grid_adapters (legacy duplicate)                         ║
║                                                                    ║
║  SIZE:                                                             ║
║  ├── Original: {original_gb:>6.2f} GB                                       ║
║  ├── New:      {new_gb:>6.2f} GB                                       ║
║  └── Change:   {change_gb:>+6.2f} GB                                        ║
║                                                                    ║
║  CAPABILITIES:                                                     ║
║  ├── ✓ Text generation (no external LLM needed)                   ║
║  ├── ✓ Audio understanding                                        ║
║  ├── ✓ Audio output (speech synthesis)                            ║
║  ├── ✓ Vision understanding (images + video)                      ║
║  ├── ✓ Grid reasoning (ARC-AGI)                                   ║
║  └── ✓ No runtime downloads required                              ║
║                                                                    ║
║  OUTPUT:                                                           ║
║  └── {str(APEX_PATH):56} ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

print(f"\nThe model is now FULLY SELF-CONTAINED.")
print(f"No external downloads. No external LLM. True multimodal voice.")

log.success("Phase Voice complete — FLUX is now self-contained!")

---

## Optional: Test Voice Generation

This requires the FLUXVoiceOmni runtime class which loads and initializes the embedded voice module.

In [ ]:
# Optional: Quick test if voice module class is available
try:
    from phases.phase_voice.voice_module import FLUXVoiceOmni
    
    print(f'\n  Loading voice module for testing...')
    
    # Load the saved model
    apex = torch.load(str(APEX_PATH), map_location='cpu', weights_only=False)
    
    # Initialize voice module
    voice = FLUXVoiceOmni(apex['voice'], device=device)
    
    print(f'  ✓ FLUXVoiceOmni initialized')
    print(f'    Base model: {voice.base_model}')
    print(f'    Hidden dim: {voice.voice_hidden_dim}')
    
    # Test bridge projections
    print(f'\n  Testing wave ↔ voice bridges:')
    dummy_wave = torch.randn(1, 10, 432)  # [batch, seq, wave_dim]
    voice_hidden = voice.wave_to_voice(dummy_wave)
    wave_back = voice.voice_to_wave(voice_hidden)
    print(f'    Wave → Voice: {dummy_wave.shape} → {voice_hidden.shape}')
    print(f'    Voice → Wave: {voice_hidden.shape} → {wave_back.shape}')
    print(f'  ✓ Bridge projections work')
    
except ImportError as e:
    print(f'  ⚠ Voice module class not available: {e}')
    print(f'    Voice weights are saved, but runtime class needs phases/phase_voice/voice_module.py')
except Exception as e:
    print(f'  ⚠ Test error: {e}')